# 🔬 01. Exploratory Data Analysis (EDA)
### Customer Segmentation & Churn Intelligence Platform

This notebook performs exploratory analysis on our customer dataset: profiling distributions, analyzing target variable imbalance, generating correlation matrices, and conducting statistical significance testing (Welch's t-test, Chi-square tests).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from src import config

# Set styling configuration
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
sns.set_theme(style="darkgrid")
logger = plt.rc('figure', facecolor='#0E1117')
plt.rc('axes', facecolor='#1E222B', edgecolor='#2D3139', labelcolor='white')
plt.rc('text', color='white')
plt.rc('xtick', color='white')
plt.rc('ytick', color='white')
logger = plt.rc('grid', color='#2D3139', linestyle='--')

## 1. Data Ingest & Profiling

In [ ]:
# Load our generated features matrix
features_path = os.path.join(config.PROCESSED_DATA_DIR, "customer_feature_matrix.csv")
if not os.path.exists(features_path):
    print("Feature matrix not found. Calculating features directly...")
    from src.feature_engineering import compute_all_customer_features
    df = compute_all_customer_features()
else:
    df = pd.read_csv(features_path)

print(f"Dataset Dimension: {df.shape}")
df.head()

## 2. Target Variable Class Imbalance Check

In [ ]:
# Define churn target base
df['is_churned'] = (df['recency_days'] > config.CHURN_THRESHOLD_DAYS).astype(int)

churn_counts = df['is_churned'].value_counts()
churn_percents = df['is_churned'].value_counts(normalize=True) * 100

print("Target Class Distribution:")
print(f"Active (0): {churn_counts[0]} ({churn_percents[0]:.2f}%)")
print(f"Churned (1): {churn_counts[1]} ({churn_percents[1]:.2f}%)")

# Visualizing Class Distribution
plt.figure(figsize=(6, 4))
sns.barplot(x=churn_counts.index, y=churn_counts.values, palette=['#10B981', '#EF4444'])
plt.title('Target Variable: Churn Class Distribution (Active vs Churned)')
plt.ylabel('User Volume')
plt.xlabel('Churn Flag (0 = Active, 1 = Churned)')
plt.savefig(os.path.join(config.PLOTS_DIR, 'eda', 'class_imbalance.png'), bbox_inches='tight')
plt.show()

## 3. Correlation Heatmaps (Collinearity Detection)

In [ ]:
# Calculate correlation matrix
num_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[num_cols].corr(method='spearman')

# Renders Heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Spearman Rank Correlation Heatmap of Engineered Features')
plt.savefig(os.path.join(config.PLOTS_DIR, 'eda', 'correlation_matrix.png'), bbox_inches='tight')
plt.show()

## 4. Statistical Significance Testing
### A. Welch's t-test (Numerical variations between Churn Classes)

In [ ]:
numerical_features = ['recency_days', 'frequency_total', 'monetary_value_total', 'avg_order_value', 'purchase_interval_avg', 'satisfaction_score_avg']

print("=== Welch's t-test Results ===")
for feat in numerical_features:
    active_values = df[df['is_churned'] == 0][feat].dropna()
    churned_values = df[df['is_churned'] == 1][feat].dropna()
    
    t_stat, p_val = stats.ttest_ind(active_values, churned_values, equal_var=False)
    print(f"{feat: <25} -> t-statistic: {t_stat: .4f} | p-value: {p_val: .4e} (Significant: {p_val < 0.05})")

### B. Chi-Square Test (Demographic Categoricals vs Churn Target)

In [ ]:
categorical_features = ['city_tier', 'gender', 'device_pref', 'marital_status']

print("=== Chi-Square test Results ===")
for feat in categorical_features:
    contingency_table = pd.crosstab(df[feat], df['is_churned'])
    chi2, p_val, dof, ex = stats.chi2_contingency(contingency_table)
    print(f"{feat: <25} -> Chi2: {chi2: .4f} | p-value: {p_val: .4e} (Significant: {p_val < 0.05})")